# 03 - Recursion Fundamentals

Phase 0 of the DSA Roadmap, final notebook. Trees, backtracking, and graph DFS (Phases 2-3) are all recursion wearing different clothes -- getting the call-stack mental model solid now pays off across the rest of the syllabus.

## Base case + recursive case, made visible

Every correct recursive function needs a base case (stops the recursion) and a recursive case (reduces the problem and calls itself). Printing the call stack as it happens makes the otherwise-invisible mechanics concrete.

In [1]:
def factorial_traced(n, depth=0):
    print("  " * depth + f"factorial({n}) called")
    if n <= 1:                                       # base case
        print("  " * depth + f"factorial({n}) -> base case, returns 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)   # recursive case
    print("  " * depth + f"factorial({n}) -> returns {result}")
    return result

factorial_traced(4)

factorial(4) called
  factorial(3) called
    factorial(2) called
      factorial(1) called
      factorial(1) -> base case, returns 1
    factorial(2) -> returns 2
  factorial(3) -> returns 6
factorial(4) -> returns 24


24

Notice the shape: calls go DOWN (indentation increasing) until the base case, then returns go back UP in reverse order -- this is the call stack, made visible. Every one of those indented lines is a real stack frame sitting in memory until it returns, which is exactly the O(n) space cost from the Big-O notebook.

## Naive recursion can recompute the same work exponentially

Naive Fibonacci recomputes `fib(n-2)` many times over -- once directly, once again inside the `fib(n-1)` branch, and so on. The call count grows like `O(2^n)`. `functools.lru_cache` (Python & DSA, notebook 03) turns this into O(n) by caching, with a one-line change.

In [2]:
import time
from functools import lru_cache

def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n):
    if n <= 1:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)

t0 = time.perf_counter(); fib_naive(28); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); fib_memo(28); t_memo = time.perf_counter() - t0
print(f"naive fib(28):    {t_naive:.4f}s")
print(f"memoized fib(28): {t_memo:.6f}s")
print(f"speedup: {t_naive/t_memo:.0f}x, from caching alone -- this IS the top-down DP pattern from Phase 4")

naive fib(28):    0.0356s
memoized fib(28): 0.000054s
speedup: 657x, from caching alone -- this IS the top-down DP pattern from Phase 4


## Tree recursion: more than one recursive call per invocation

Fibonacci above branches into two recursive calls per invocation -- exactly the shape of recursion on a binary tree (Phase 2). Counting nodes in a nested list structure is a simpler warm-up for the same "recurse into children, combine results" pattern.

In [3]:
def count_leaves(nested):
    if not isinstance(nested, list):     # base case: a plain value, not a list
        return 1
    return sum(count_leaves(item) for item in nested)   # recursive case: sum over children

data = [1, [2, 3], [4, [5, 6, [7]]], 8]
print("leaf count:", count_leaves(data))   # 1,2,3,4,5,6,7,8 -> 8 leaves

leaf count: 8


## Converting recursion to iteration with an explicit stack

Any recursive traversal can be rewritten iteratively by managing a stack manually instead of relying on the call stack -- useful when recursion depth risks hitting Python's limit (below), and it is exactly how iterative DFS is written in Phase 2-3.

In [4]:
def count_leaves_iterative(nested):
    stack = [nested]
    count = 0
    while stack:
        item = stack.pop()
        if isinstance(item, list):
            stack.extend(item)          # push children instead of recursing into them
        else:
            count += 1
    return count

print("iterative leaf count:", count_leaves_iterative(data))
print("matches recursive version:", count_leaves_iterative(data) == count_leaves(data))

iterative leaf count: 8
matches recursive version: True


## Python has no tail-call optimization -- a real, practical limit

Some languages optimize a recursive call in "tail position" (the very last thing a function does) into a loop, using no extra stack space. CPython deliberately does not do this -- every recursive call costs a real stack frame, tail position or not, and the default recursion limit (~1000) is a real ceiling in practice.

In [5]:
import sys

def tail_recursive_sum(n, acc=0):
    if n == 0:
        return acc
    return tail_recursive_sum(n - 1, acc + n)     # tail position -- but Python still stacks it

print("works fine at n=500:", tail_recursive_sum(500))

sys.setrecursionlimit(3000)
try:
    tail_recursive_sum(5000)
    print("succeeded at n=5000")
except RecursionError as e:
    print("RecursionError at n=5000:", str(e)[:60])
    print("-> the fix is the SAME as above: rewrite as an explicit loop, not a deeper recursion limit")

works fine at n=500: 125250
RecursionError at n=5000: maximum recursion depth exceeded
-> the fix is the SAME as above: rewrite as an explicit loop, not a deeper recursion limit


This is a genuine, common interview-relevant gotcha: a recursive solution that is *correct* can still fail at scale in Python specifically, in a way it might not in a language with tail-call optimization -- worth mentioning out loud (Framework step 10) when proposing a deeply recursive approach on a problem with a large input bound.

## Practice

Implement each TODO, then run the check cell.

In [6]:
def reverse_string_recursive(s):
    # Return s reversed, using recursion (not slicing tricks like s[::-1]).
    raise NotImplementedError

def sum_list_recursive(nums):
    # Return the sum of nums, using recursion, not sum()/a loop.
    raise NotImplementedError

def binary_search_recursive(nums, target, lo=0, hi=None):
    # Return the index of target in sorted list nums, or -1 if absent. Recursive, O(log n).
    raise NotImplementedError

In [7]:
def _check(label, ok, detail=""):
    print(("[PASS] " if ok else "[FAIL] ") + label, detail)

try:
    r = reverse_string_recursive("hello")
    _check("reverse_string_recursive", r == "olleh", r)
except NotImplementedError:
    print("[SKIP] reverse_string_recursive -- not implemented yet")

try:
    r = sum_list_recursive([1, 2, 3, 4, 5])
    _check("sum_list_recursive", r == 15, r)
except NotImplementedError:
    print("[SKIP] sum_list_recursive -- not implemented yet")

try:
    nums = [1, 3, 5, 7, 9, 11]
    r1 = binary_search_recursive(nums, 7)
    r2 = binary_search_recursive(nums, 4)
    _check("binary_search_recursive finds present value", r1 == 3, r1)
    _check("binary_search_recursive returns -1 for absent value", r2 == -1, r2)
except NotImplementedError:
    print("[SKIP] binary_search_recursive -- not implemented yet")

[SKIP] reverse_string_recursive -- not implemented yet
[SKIP] sum_list_recursive -- not implemented yet
[SKIP] binary_search_recursive -- not implemented yet


## Self-check before moving on

- [ ] I can identify the base case and recursive case in any recursive function I read
- [ ] I can explain why naive Fibonacci is O(2^n) and how memoization fixes it
- [ ] I can convert a simple recursive traversal into an iterative one using an explicit stack
- [ ] I know Python has no tail-call optimization and what that means practically for deep recursion
- [ ] I count recursion call-stack depth as space complexity, automatically, not as an afterthought

Phase 0 complete. Next: **Phase 1 — Foundational Patterns**, starting with `01-arrays-hashing/`.